In [15]:
# -----------------------------------------------------------------------------
# 1. LOAD REQUIRED LIBRARIES
# -----------------------------------------------------------------------------
library(DESeq2)
library(dplyr)
library(tibble)
library(org.Hs.eg.db)
library(AnnotationDbi)
library(KEGGREST)

In [16]:
# -----------------------------------------------------------------------------
# 2a. LOAD CURATED GENE LIST
# -----------------------------------------------------------------------------
filename    <- "input/names_genes_dna_repair.csv"
genes_names <- read.csv(filename)

# Separate genes with multiple pathways (e.g. "BER/MMR") into individual rows
genes_names <- genes_names %>%
  mutate(pathway = strsplit(pathway, "/")) %>%
  tidyr::unnest(pathway) %>%
  mutate(pathway = trimws(pathway))

  # -----------------------------------------------------------------------------
# 2b. MAP CURATED PATHWAYS TO OFFICIAL KEGG NAMES
# -----------------------------------------------------------------------------
# Define KEGG IDs for your curated pathway abbreviations
# Update this list if you have more pathways in your CSV
kegg_pathway_ids <- c(
  "BER" = "hsa03410",
  "MMR" = "hsa03430",
  "NER" = "hsa03420",
  "HR"  = "hsa03440"
)

# Fetch official names from KEGG
kegg_official_names <- sapply(kegg_pathway_ids, function(id){
  keggGet(id)[[1]]$NAME
})

# Build reference table
kegg_reference <- data.frame(
  curated_pathway = names(kegg_pathway_ids),
  kegg_id         = kegg_pathway_ids,
  official_name   = kegg_official_names,
  row.names       = NULL
)

In [4]:
dds_k6_8 <- readRDS("output/DE_k6-K8.rds")

In [5]:
# -----------------------------------------------------------------------------
# 3. EXTRACT NORMALIZED COUNTS FROM DESEQ2 OBJECT
# -----------------------------------------------------------------------------
# Get size-factor normalized counts
norm_counts <- counts(dds_k6_8, normalized = TRUE)

# Calculate mean normalized expression per group
# condition 2 = cells WITH SHM events
# condition 1 = cells WITHOUT SHM events
mean_with_SHM    <- rowMeans(norm_counts[, colData(dds_k6_8)$condition == "2"])
mean_without_SHM <- rowMeans(norm_counts[, colData(dds_k6_8)$condition == "1"])

# Combine into a dataframe
norm_counts_df <- data.frame(
  rowname          = rownames(norm_counts),
  mean_with_SHM    = mean_with_SHM,
  mean_without_SHM = mean_without_SHM
)

In [7]:
DE_scSHM <- read.csv("output/DE_results_k6-8.csv" )

In [8]:
# -----------------------------------------------------------------------------
# 4. PREPARE DE RESULTS TABLE
# -----------------------------------------------------------------------------
# Map Entrez IDs and gene full names
DE_scSHM$entrez <- mapIds(org.Hs.eg.db,
                          keys     = DE_scSHM$rowname,
                          column   = "ENTREZID",
                          keytype  = "SYMBOL",
                          multiVals = "first")

DE_scSHM$name <- mapIds(org.Hs.eg.db,
                        keys     = DE_scSHM$rowname,
                        column   = "GENENAME",
                        keytype  = "SYMBOL",
                        multiVals = "first")

# Add normalized expression per group to DE results
DE_scSHM <- DE_scSHM %>%
  left_join(norm_counts_df, by = "rowname")

'select()' returned 1:1 mapping between keys and columns

'select()' returned 1:1 mapping between keys and columns



In [11]:
# -----------------------------------------------------------------------------
# 5. FILTER FOR DNA REPAIR GENES AND ADD PATHWAY ANNOTATION
# -----------------------------------------------------------------------------
DE_filtered <- DE_scSHM %>%
  inner_join(genes_names, by = c("rowname" = "gene"))

In [17]:
# -----------------------------------------------------------------------------
# 7. BUILD FINAL TABLE
# -----------------------------------------------------------------------------
final_table <- DE_filtered %>%
  left_join(kegg_reference, by = c("pathway" = "curated_pathway")) %>%
  dplyr::select(
    Gene             = rowname,
    Full_name        = name,
    Curated_pathway  = pathway,
    KEGG_id          = kegg_id,
    Official_name    = official_name,
    mean_with_SHM,
    mean_without_SHM,
    log2FoldChange,
    padj
  ) %>%
  arrange(Curated_pathway, padj)

# Preview
head(final_table)

,Gene,Full_name,Curated_pathway,KEGG_id,Official_name,mean_with_SHM,mean_without_SHM,log2FoldChange,padj
,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
1,HMGB2,high mobility group box 2,BER,hsa03410,Base excision repair - Homo sapiens (human),2.55291715,1.7689924,0.5121202,0.00019509
2,UNG,uracil DNA glycosylase,BER,hsa03410,Base excision repair - Homo sapiens (human),0.17235276,0.1304895,0.5507903,0.01660639
3,PARP3,poly(ADP-ribose) polymerase family member 3,BER,hsa03410,Base excision repair - Homo sapiens (human),0.07319167,0.1046078,-0.8009021,0.01941252
4,HMGB1,high mobility group box 1,BER,hsa03410,Base excision repair - Homo sapiens (human),7.89905595,7.1948021,0.1670893,0.06202366
5,POLE3,"DNA polymerase epsilon 3, accessory subunit",BER,hsa03410,Base excision repair - Homo sapiens (human),0.36904281,0.2771857,0.3172805,0.08379320
6,PARP4,poly(ADP-ribose) polymerase family member 4,BER,hsa03410,Base excision repair - Homo sapiens (human),0.10491275,0.1522067,-0.4942571,0.12056632


In [18]:
# Save as CSV
write.csv(final_table, 
          file = "output/DE_DNA_repair_genes.csv", 
          row.names = FALSE)

In [20]:
DE_filtered


rowname,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,entrez,name,mean_with_SHM,mean_without_SHM,pathway
<chr>,<dbl>,<dbl>,<lgl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<chr>
RPA2,0.37296983,0.306739578,NA,8.976331696,2.739831e-03,4.514373e-02,6118,replication protein A2,0.45489850,0.37111308,MMR
MUTYH,0.12027248,0.015169434,NA,0.006478084,9.358515e-01,9.825120e-01,4595,mutY DNA glycosylase,0.15040480,0.11958959,BER
ADAR,0.59146687,-0.146156004,NA,2.317313788,1.279636e-01,4.590794e-01,103,adenosine deaminase RNA specific,0.50509922,0.59342423,RNSA_editing
PARP1,3.52315493,0.103461795,NA,4.840284592,2.781875e-02,2.022428e-01,142,poly(ADP-ribose) polymerase 1,3.57523408,3.52197465,BER
MSH2,0.19125059,0.311121975,NA,5.308098992,2.124123e-02,1.734491e-01,4436,mutS homolog 2,0.22249087,0.19054258,MMR
MSH6,0.28189163,0.220133659,NA,3.485399382,6.193352e-02,3.159609e-01,2956,mutS homolog 6,0.30858847,0.28128660,MMR
POLE4,0.46195374,-0.055376264,NA,0.310004685,5.776863e-01,8.430729e-01,56655,"DNA polymerase epsilon 4, accessory subunit",0.43365815,0.46259501,BER
PMS1,0.24293916,-0.343860789,NA,5.313040876,2.118108e-02,1.734491e-01,5378,"PMS1 homolog 1, mismatch repair system component",0.20437943,0.24381305,MMR
OGG1,0.27435330,0.025913992,NA,0.044935067,8.321266e-01,9.542121e-01,4968,8-oxoguanine DNA glycosylase,0.27524908,0.27433300,BER


In [23]:
# -----------------------------------------------------------------------------
# SEARCH KEGG PATHWAYS FOR SHM AND RNA EDITING GENES
# -----------------------------------------------------------------------------

# Filter genes from those curated groups
genes_to_search <- DE_filtered %>%
  filter(pathway %in% c("SHM", "RNSA_editing")) %>%
  dplyr::select(rowname, entrez, pathway) %>%
  distinct()

# Query KEGG for each gene
kegg_results <- lapply(genes_to_search$entrez, function(entrez_id){
  tryCatch({
    pathways <- keggGet(paste0("hsa:", entrez_id))[[1]]$PATHWAY
    if(!is.null(pathways)){
      data.frame(
        entrez        = entrez_id,
        kegg_id       = names(pathways),
        official_name = as.character(pathways)
      )
    }
  }, error = function(e) NULL)
})

# Combine results
kegg_results_df <- bind_rows(kegg_results) %>%
  left_join(genes_to_search, by = "entrez")

# Preview
kegg_results_df %>% 
  dplyr::select(rowname, pathway, kegg_id, official_name)

rowname,pathway,kegg_id,official_name
<chr>,<chr>,<chr>,<chr>
ADAR,RNSA_editing,hsa04622,RIG-I-like receptor signaling pathway
ADAR,RNSA_editing,hsa04623,Cytosolic DNA-sensing pathway
ADAR,RNSA_editing,hsa05162,Measles
ADAR,RNSA_editing,hsa05164,Influenza A
ADAR,RNSA_editing,hsa05171,Coronavirus disease - COVID-19
AICDA,SHM,hsa04672,Intestinal immune network for IgA production
AICDA,SHM,hsa05340,Primary immunodeficiency
